# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We'll use the FAIR² dataset provided via a [Croissant schema](https://croissant.readthedocs.io/) URL, access its contents by `@id`, and perform typical data analysis steps.

### Dataset Source
- Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- Title: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells

In [ ]:
# Install `mlcroissant` if not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and access core information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Dataset
dataset = mlc.Dataset(url)
print(f"Loaded Croissant metadata from: {url}")
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Let's review available record sets and their schema, referencing all entities by their `@id`.

In [ ]:
# List available record sets
record_set_ids = [rset['@id'] for rset in dataset.metadata.record_sets]
if not record_set_ids:
    print("No record sets found in this dataset metadata.")
else:
    print("Available record sets (@id):")
    for rid in record_set_ids:
        print(f"  - {rid}")

# For each record set, list fields and columns
for recset in dataset.metadata.record_sets:
    print(f"\nRecordSet @id: {recset['@id']}")
    fields = recset.get('field', [])
    if not fields:
        print("  No fields found.")
    else:
        print("  Fields (by @id):")
        for f in fields:
            # f can be dict or string (if compacted)
            if isinstance(f, dict):
                print(f"    - {f['@id']}")
            else:
                print(f"    - {f}")

    # For each field, show associated column(s) if any
    for f in fields:
        field_dict = None
        # Lookup actual field definition in metadata.fields
        # Support both compact and expanded forms
        if isinstance(f, str):
            candidates = [field for field in getattr(dataset.metadata, 'fields', []) if field.get('@id') == f]
            if candidates:
                field_dict = candidates[0]
        elif isinstance(f, dict):
            field_dict = f
        if field_dict is not None:
            columns = field_dict.get('column', [])
            if columns:
                print(f"      Columns for field {field_dict['@id']}:")
                for col in columns:
                    # col may be a string or dict
                    if isinstance(col, dict):
                        print(f"        - {col['@id']}")
                    else:
                        print(f"        - {col}")

## 3. Data Extraction
Load one or more record sets into Pandas DataFrame. All entities referenced by `@id`.

**Tip:** You can find record set and field `@id`s from the above overview.

In [ ]:
# Example: extract data from the main record set by its @id
#
# If the above section showed one or more record set IDs, use them here.
# For demonstration, we attempt to load the record sets if present.

dataframes = {}
for rset_id in record_set_ids:
    print(f"Extracting records for @id: {rset_id}")
    records = list(dataset.records(record_set=rset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print("  No records found for this record set.")

# Use the first record set as the working example for further analysis
if record_set_ids:
    working_set_id = record_set_ids[0]
    df = dataframes[working_set_id]
else:
    working_set_id = None
    df = None

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps like filtering, normalization, and grouping. All field access by `@id`.

> Adjust `numeric_field_id` and `group_field_id` to match those listed in your record set.

In [ ]:
# Edit these IDs to match your dataset fields! Example IDs as placeholders below.
numeric_field_id = None  # e.g. 'cr:MW' or the @id of a numeric field
group_field_id = None    # e.g. 'cr:accession' or the @id of a grouping field

# Try to infer from DataFrame columns if present
if df is not None and df.shape[1] > 0:
    for c in df.columns:
        # Try to pick a likely numeric field
        if numeric_field_id is None and pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
        # Try to pick a categorical/group field
        if group_field_id is None and pd.api.types.is_string_dtype(df[c]):
            group_field_id = c
        if numeric_field_id and group_field_id:
            break
if numeric_field_id is None or group_field_id is None:
    print("Could not auto-identify numeric or group field. Please set their @id explicitly based on data overview.")

if df is not None and numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} (mean):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields. Access fields by column `@id`.

Example: plot distribution of a numeric field or scatter plot by another field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # Example: scatter plot if another numeric field exists
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 1:
        y_field = [c for c in numeric_cols if c != numeric_field_id][0]
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=df[numeric_field_id], y=df[y_field])
        plt.xlabel(numeric_field_id)
        plt.ylabel(y_field)
        plt.title(f"{numeric_field_id} vs {y_field}")
        plt.show()

## 6. Conclusion
- We demonstrated loading a Croissant dataset by schema URL and overviewed its structure (record sets, fields, columns).
- We programmatically accessed data by referencing entity `@id` values, extracted example tables, and performed basic filtering and normalization.
- Visual exploratory analysis was conducted for a selected numeric field.

**Note:** For more detailed analysis, consult the [FAIR² schema documentation](https://github.com/mlcommons/croissant) and dataset supplement. Update field/group IDs above based on your actual data for deeper exploration!